# Milestone 1: Automated Pipeline for arXiv Metadata, TeX Sources, and References

Notebook này phục vụ việc thực thi trọn vẹn toàn bộ quy trình Milestone 1 cho từng bài báo trong khoảng `start_id` đến `end_id`:
1. **Cào Metadata** từ arXiv API -> lưu `data/papers.jsonl` và ghi thống kê **Bảng 1: `m1_discovery_stats.csv`**.
2. **Tải mã nguồn & giải nén** chỉ giữ lại các file `.tex` và `.bib` (loại bỏ hoàn toàn các file ảnh/hình nhị phân, `.sty`, `.cls`, `.pdf`) -> lưu vào `data/raw/<clean_arxiv_id>/` và ghi thống kê **Bảng 2: `m1_download_stats.csv`**.
3. **Cào danh sách trích dẫn (References)** từ Semantic Scholar Graph API -> lưu `data/references.jsonl` và ghi thống kê **Bảng 3: `m1_reference_stats.csv`**.
4. **Đo lường Benchmark Hệ thống** tự động trong suốt quá trình chạy -> ghi thống kê **Bảng 4: `m1_system_benchmark.csv`**.

### Cơ chế chống lỗi mạng & quá tải (Rate limit & Retry):
- Tuân thủ khoảng cách tối thiểu giữa các request: $\ge 3.0$s với arXiv, $\ge 1.0$s với Semantic Scholar.
- Tự động retry với exponential backoff khi gặp lỗi HTTP 429 (Too Many Requests), HTTP 500/502/503/504, và sự cố gián đoạn kết nối mạng.
- Fail fast không retry với HTTP 404.
- Mỗi request lỗi tạm thời được gọi tối đa **5 lần tổng cộng**; hết lượt thì ghi lỗi và chuyển sang ID tiếp theo.
- Hỗ trợ Resume: bỏ qua kết quả hoàn tất còn hợp lệ; lỗi tạm thời được thử lại ở lần chạy sau khi hết thời gian chờ (60 giây, tăng đôi, tối đa 1 giờ).
- Trạng thái của từng bước lưu trong `data/csv/*.ledger.jsonl`; CSV thống kê giữ nguyên các cột hiện có.

## 1. Kết nối Google Drive và Thiết lập Môi trường (trên Colab)

In [ ]:
# Chạy cell này nếu bạn thực thi trên Google Colab
try:
    from google.colab import drive
    import os

    drive.mount('/content/drive')

    # Thư mục lưu trữ dự án trên Google Drive của bạn
    PROJECT_DIR = '/content/drive/MyDrive/Colab_Notebooks/arXiv-Project'

    if not os.path.exists(PROJECT_DIR):
        print(f"Thư mục {PROJECT_DIR} chưa tồn tại, đang clone từ GitHub...")
        !git clone -b fix/pipeline-recovery https://github.com/ahtuan171/arXiv-Project.git "{PROJECT_DIR}"
    else:
        print("Đang cập nhật mã nguồn mới nhất từ GitHub...")
        !git -C "{PROJECT_DIR}" pull origin fix/pipeline-recovery || echo "!! CẢNH BÁO: git pull thất bại - đang chạy với mã nguồn hiện tại"

    os.chdir(PROJECT_DIR)
    print("Thư mục làm việc hiện tại:", os.getcwd())
except ImportError:
    print("Đang chạy ở môi trường cục bộ (Local).")

In [ ]:
# Cài đặt các thư viện cần thiết
%pip install -r requirements.txt


### 1.1. Cấu hình Semantic Scholar API Key (Tùy chọn)

Để cào danh sách trích dẫn (References) với quota cao hơn (100 requests / 5 phút thay vì quota ẩn danh), bạn có thể nạp API Key từ **Google Colab Secrets**:
1. Nhấn vào biểu tượng **Secrets** (🔑) ở thanh công cụ bên trái giao diện Colab.
2. Bấm **"Add new secret"**, đặt tên bí mật là `S2_API_KEY` và dán API Key vào.
3. Bật quyền **Notebook access** cho secret `S2_API_KEY`.

In [ ]:
import os
import requests
from src import config

# Lấy S2_API_KEY từ Google Colab Secrets (hoặc biến môi trường nếu chạy cục bộ)
try:
    from google.colab import userdata
    s2_key = userdata.get('S2_API_KEY')
except (ImportError, Exception):
    s2_key = os.environ.get('S2_API_KEY', '')

if s2_key:
    s2_key = s2_key.strip()
    os.environ["S2_API_KEY"] = s2_key
    config.S2_API_KEY = s2_key
    print(f"✅ Đã nạp S2_API_KEY thành công: {s2_key[:4]}...{s2_key[-4:]} (độ dài: {len(s2_key)})")

    # Kiểm tra tính hợp lệ của key bằng một request thử nghiệm
    try:
        test_url = "https://api.semanticscholar.org/graph/v1/paper/arXiv:2502.13235?fields=title"
        res = requests.get(test_url, headers={"x-api-key": s2_key}, timeout=10)
        if res.status_code == 200:
            print("🎉 Xác thực Semantic Scholar API thành công! (HTTP 200 OK)")
        elif res.status_code in (401, 403):
            print(f"❌ CẢNH BÁO: Key không hợp lệ hoặc không có quyền (HTTP {res.status_code}): {res.text}")
        else:
            print(f"ℹ️ Phản hồi từ Semantic Scholar API: HTTP {res.status_code}")
    except Exception as err:
        print(f"⚠️ Không thể gửi request kiểm tra API: {err}")
else:
    print("⚠️ Chưa tìm thấy S2_API_KEY. Pipeline sẽ chạy ở chế độ ẩn danh (quota thấp hơn).")
    print("💡 Mẹo: Thêm secret 'S2_API_KEY' trong mục Secrets (biểu tượng chìa khóa 🔑) ở thanh bên trái Colab.")


## 2. Khai báo khoảng arXiv-ID cần xử lý

In [ ]:
#@title Cấu hình tham số Pipeline (Khoảng ID, Limit & Tùy chọn Retry) { display-mode: "form" }

#@markdown ### 1. Khoảng arXiv ID và Giới hạn
#@markdown Nhập `start_id` và `end_id` theo định dạng `YYMM.NNNNN` (ví dụ: `2502.13235`).
start_id = "2502.13235"  #@param {type:"string"}
end_id = "2502.13245"    #@param {type:"string"}

#@markdown Giới hạn số bài báo chạy thử nghiệm (đặt `0` để chạy toàn bộ dải ID):
LIMIT = 0                #@param {type:"integer"}

#@markdown ### 2. Tùy chọn Phục hồi & Retry (Recovery Options)
#@markdown - **resume**: Bỏ qua các ID đã hoàn thành hợp lệ (khuyến nghị giữ `True`).
#@markdown - **retry_errors**: Thử lại ngay các ID từng gặp lỗi (lỗi mạng, HTTP 429/5xx).
#@markdown - **retry_not_found**: Thử lại các ID từng trả về HTTP 404 Not Found.
#@markdown - **retry_partial**: Cào bổ sung trích dẫn chưa đầy đủ (chỉ áp dụng Bước 3).
resume = True            #@param {type:"boolean"}
retry_errors = False     #@param {type:"boolean"}
retry_not_found = False  #@param {type:"boolean"}
retry_partial = False    #@param {type:"boolean"}

#@markdown ### 3. Tối ưu Hiệu năng (Performance Optimization)
#@markdown - **parallel_execution**: Chạy đồng thời Bước 2 (Tải mã nguồn) và Bước 3 (Cào trích dẫn Semantic Scholar).
#@markdown - **batch_size_metadata**: Gom cụm truy vấn arXiv Metadata (mặc định 100 ID/request).
parallel_execution = True  #@param {type:"boolean"}
batch_size_metadata = 100  #@param {type:"integer"}

# Dùng chung cho cả ba bước. Giữ resume=True để phục hồi lỗi, không tải lại tất cả.
RECOVERY_OPTIONS = {
    "resume": resume,
    "retry_errors": retry_errors,
    "retry_not_found": retry_not_found,
    "retry_partial": retry_partial,
}

from src.utils import parse_arxiv_range
id_list = parse_arxiv_range(start_id, end_id)
if LIMIT > 0:
    id_list = id_list[:LIMIT]

if not id_list:
    raise ValueError("id_list rỗng: vui lòng kiểm tra lại start_id và end_id")

print(f"-> Tổng số bài báo sẽ xử lý: {len(id_list)}")
print("ID đầu tiên:", id_list[0] if id_list else "None")
print("ID cuối cùng:", id_list[-1] if id_list else "None")
print("Tùy chọn phục hồi (RECOVERY_OPTIONS):", RECOVERY_OPTIONS)
print(f"Chế độ chạy song song (parallel_execution): {parallel_execution}")
print(f"Kích thước batch Metadata (batch_size_metadata): {batch_size_metadata}")


## 3. Khởi động Bộ theo dõi Benchmark Hệ thống (Bảng 4)

In [ ]:
from pathlib import Path
from src import config, utils
from src.benchmark import SystemBenchmarkTracker

# Kích hoạt logging ra file m1_pipeline.log
utils.setup_logging("m1_pipeline", config.LOG_DIR / "m1_pipeline.log")

# Các thư mục lưu trữ theo đúng quy định:
# Nguồn giải nén lưu tại data/raw, thống kê CSV lưu tại data/csv
RAW_DIR = config.DATA_DIR / "raw"
CSV_DIR = config.DATA_DIR / "csv"
RAW_DIR.mkdir(parents=True, exist_ok=True)
CSV_DIR.mkdir(parents=True, exist_ok=True)

# Khởi động background resource tracker (đo RAM, Disk và Wall Time)
# disk_sample_interval_sec=30.0 giúp tối ưu I/O trên Google Drive tránh hiện tượng lag/freeze
tracker = SystemBenchmarkTracker(
    storage_dirs=[config.DATA_DIR],
    sample_interval_sec=0.5,
    disk_sample_interval_sec=30.0,
)
tracker.start()
print("-> Đã kích hoạt SystemBenchmarkTracker để theo dõi Bảng 4 (m1_system_benchmark.csv).")


## 4. Thực thi Toàn bộ Pipeline Thu thập & Đo lường (Bước 1 - Bước 4)

Thực thi 4 bước thu thập theo đúng quy trình và bọc trong khối `try...finally` để đảm bảo Bảng 4 luôn được xuất:
- **Bước 1 (Bảng 1):** Cào Metadata từ arXiv API (`m1_discovery_stats`) với cơ chế gom nhóm (Batching 100 ID/request) siêu tốc.
- **Bước 2 (Bảng 2) & Bước 3 (Bảng 3):** Tải và trích xuất file nguồn TeX (`m1_download_stats`) đồng thời với Cào danh sách trích dẫn từ Semantic Scholar (`m1_reference_stats`) qua cơ chế đa luồng an toàn.
- **Bước 4 (Bảng 4):** Dừng tracker và lưu báo cáo hiệu năng hệ thống (`m1_system_benchmark`).


In [ ]:
from src.metadata_module import crawl_papers
from src.download_module.downloader import download_papers_range
from src.crawl_references_module import crawl_references_range

stats_papers = {}
actual_papers_count = len(id_list)
try:
    print(f"=== BẮT ĐẦU CÀO METADATA BÀI BÁO (arXiv API - Batch size: {batch_size_metadata}) ===")
    stats_papers = crawl_papers(
        id_list=id_list,
        out_path=config.DATA_DIR / "papers.jsonl",
        csv_path=CSV_DIR / "m1_discovery_stats.csv",
        batch_size=batch_size_metadata,
        **RECOVERY_OPTIONS,
    )
    print("-> Kết quả cào Metadata:", stats_papers)

    if parallel_execution:
        print("\n=== BẮT ĐẦU THỰC THI ĐỒNG THỜI: TẢI SOURCE VÀ CÀO TRÍCH DẪN ===")
        from concurrent.futures import ThreadPoolExecutor

        with ThreadPoolExecutor(max_workers=2) as executor:
            future_download = executor.submit(
                download_papers_range,
                id_list=id_list,
                raw_dir=RAW_DIR,
                csv_dir=CSV_DIR,
                delay=config.ARXIV_MIN_INTERVAL,
                **RECOVERY_OPTIONS,
                show_progress=True,
            )
            future_refs = executor.submit(
                crawl_references_range,
                id_list=id_list,
                out_path=config.DATA_DIR / "references.jsonl",
                csv_path=CSV_DIR / "m1_reference_stats.csv",
                **RECOVERY_OPTIONS,
            )

            df_downloads = future_download.result()
            df_refs = future_refs.result()

        print(f"\n-> Đã ghi thống kê tải mã nguồn cho {len(df_downloads)} ID (gồm cả lỗi).")
        print(df_downloads["download_status"].value_counts())
        print(f"\n-> Đã ghi thống kê trích dẫn cho {len(df_refs)} ID (gồm cả lỗi).")
        print(df_refs["ref_status"].value_counts())
    else:
        print("\n=== BẮT ĐẦU TẢI VÀ GIẢI NÉN MÃ NGUỒN (.tex, .bib ONLY) ===")
        df_downloads = download_papers_range(
            id_list=id_list,
            raw_dir=RAW_DIR,
            csv_dir=CSV_DIR,
            delay=config.ARXIV_MIN_INTERVAL,
            **RECOVERY_OPTIONS,
            show_progress=True,
        )
        print(f"-> Đã ghi thống kê tải mã nguồn cho {len(df_downloads)} ID (gồm cả lỗi).")
        print(df_downloads["download_status"].value_counts())

        print("\n=== BẮT ĐẦU CÀO DANH SÁCH TRÍCH DẪN (Semantic Scholar API) ===")
        df_refs = crawl_references_range(
            id_list=id_list,
            out_path=config.DATA_DIR / "references.jsonl",
            csv_path=CSV_DIR / "m1_reference_stats.csv",
            **RECOVERY_OPTIONS,
        )
        print(f"-> Đã ghi thống kê trích dẫn cho {len(df_refs)} ID (gồm cả lỗi).")
        print(df_refs["ref_status"].value_counts())
finally:
    print("\n=== KẾT THÚC VÀ LƯU BENCHMARK HỆ THỐNG ===")
    df_benchmark = tracker.stop(
        total_papers=actual_papers_count,
        csv_path=CSV_DIR / "m1_system_benchmark.csv",
    )
    print("-> Bảng 4: m1_system_benchmark:")
    display(df_benchmark)


### Chạy lại và phục hồi các ID bị lỗi

1. Sau khi runtime gián đoạn, thiết lập lại môi trường và chạy các bước với `resume=True`. Dữ liệu cũ được nhận diện tự động; không cần xóa JSONL/CSV hay thư mục source.
2. Nếu lỗi vừa xảy ra, đợi hết thời gian chờ rồi chạy lại ba cell thu thập. Hệ thống chỉ thử những ID chưa hoàn tất và lỗi đã đến hạn; không có tác vụ retry chạy nền.
3. Muốn thử ngay lỗi mạng hoặc sau khi sửa API key/quyền, đổi `RECOVERY_OPTIONS["retry_errors"] = True` trước khi chạy lại ba cell. Cờ này không tải lại kết quả thành công; đặt về `False` cho lần chạy thông thường.
4. Chỉ bật `retry_not_found` hoặc `retry_partial` khi cần kiểm tra lại nhóm tương ứng. `pdf_only` là kết quả đã kiểm tra không có source TeX, không phải lỗi mạng.

Đọc dòng mới nhất của mỗi ID trong `m1_discovery_stats.ledger.jsonl`, `m1_download_stats.ledger.jsonl` và `m1_reference_stats.ledger.jsonl` để xem `status`, `error_type`, `error`, `attempts` và `next_retry_at` (Unix timestamp). `failed` trong CSV vẫn là lỗi, không có nghĩa ID đã tải thành công. Với references, trạng thái `partial` nằm trong ledger; CSV giữ `success` cho truy vấn đã trả kết quả.

Metadata dùng journal để phục hồi khi dừng trước lúc hợp nhất snapshot; source dùng thư mục tạm và backup khi công bố kết quả. Không xóa các file phục hồi khi một lượt đang chạy. Chi tiết xem [hướng dẫn pipeline](README.md#q3-các-id-từng-tải-lỗi-có-được-thử-lại-không).


## 5. Kiểm tra và Trích xuất Toàn bộ 4 Bảng Thống kê Milestone 1

In [ ]:
import pandas as pd

print("=================================================================")
print("TỔNG HỢP CÁC BẢNG DỮ LIỆU MILESTONE 1 (LƯU TẠI data/csv/)")
print("=================================================================")

t1_file = CSV_DIR / "m1_discovery_stats.csv"
t2_file = CSV_DIR / "m1_download_stats.csv"
t3_file = CSV_DIR / "m1_reference_stats.csv"
t4_file = CSV_DIR / "m1_system_benchmark.csv"

if t1_file.exists():
    print("\n--- 1. BẢNG 1: m1_discovery_stats (Khám phá Metadata) ---")
    df1 = pd.read_csv(t1_file, dtype={"arxiv_id": str})
    display(df1.head(10))

if t2_file.exists():
    print("\n--- 2. BẢNG 2: m1_download_stats (Tải nguồn TeX / Bib) ---")
    df2 = pd.read_csv(t2_file, dtype={"arxiv_id": str})
    display(df2.head(10))

if t3_file.exists():
    print("\n--- 3. BẢNG 3: m1_reference_stats (Trích dẫn Semantic Scholar) ---")
    df3 = pd.read_csv(t3_file, dtype={"arxiv_id": str})
    display(df3.head(10))

if t4_file.exists():
    print("\n--- 4. BẢNG 4: m1_system_benchmark (Benchmark CPU/RAM/Disk trên Colab) ---")
    df4 = pd.read_csv(t4_file)
    display(df4)